# Long short-term memory: a hands-on implementation

**Track:** Neural networks · **Guide:** [`022-lstm.md`](../guides/concepts/022-lstm.md)

An LSTM is a gated RNN designed to preserve or forget information over longer sequences.

> **Intuition:** A gated notebook chooses which facts to keep, erase, or expose at every step.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Long short-term memory]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** A sequence model must retain an early warning signal through many ordinary sensor readings.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Define the LSTM gates

An LSTM keeps a cell memory and hidden state. Sigmoid gates choose what to forget, write, and expose.


In [ ]:
import numpy as np


def sigmoid(values):
    return 1 / (1 + np.exp(-np.asarray(values, dtype=float)))

def lstm_step(x, h_previous, c_previous, parameters):
    gates = parameters["W"] @ np.array([x, h_previous]) + parameters["b"]
    forget, write, candidate, expose = np.split(gates, 4)
    forget, write, expose = sigmoid(forget), sigmoid(write), sigmoid(expose)
    candidate = np.tanh(candidate)
    cell = forget * c_previous + write * candidate
    hidden = expose * np.tanh(cell)
    return hidden.item(), cell.item()

## 2. Process a sequence with memory

Small fixed weights make the gates executable and inspectable. Training weights is a separate optimization task.


In [ ]:
parameters = {
    "W": np.array([
        [0.4, 0.2], [0.7, 0.1], [0.8, -0.2], [0.5, 0.3],
    ]),
    "b": np.zeros(4),
}
values = [1.0, 0.0, 0.0, 0.0, 0.0]
h, c = 0.0, 0.0
history = []
for timestep, value in enumerate(values):
    h, c = lstm_step(value, h, c, parameters)
    history.append((timestep, value, round(h, 3), round(c, 3)))
print("step, input, hidden, cell:", history)

## 3. Compare retained cell memory

The cell state carries information separately from the exposed hidden state, helping gradients and information persist over longer sequences.


In [ ]:
print("final cell memory:", history[-1][3])
assert np.isfinite(history[-1][3])
print("The weights are illustrative; a real LSTM learns them from sequence data.")

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/022-lstm.md) and the
[learning roadmap](../guides/ROADMAP.md).
